# Inspect `data/processed/` parquet files

Quick structural look at each file: shape, dtypes, memory, null counts, head, and basic numeric summary. Also peeks at the non-parquet artifacts (`_watermark.json`, `scaler.pkl`, `umap_emb.npy`).

In [1]:
from pathlib import Path
import json, pickle

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

# Adjust if your notebook lives elsewhere
DATA = Path('data/processed')
assert DATA.exists(), f'Not found: {DATA.resolve()}'
sorted(p.name for p in DATA.iterdir())

AssertionError: Not found: /Users/ck/Downloads/data/processed

## Schema-only pass (cheap)

Reads just the parquet footer — no row data loaded. Good first scan to see column counts and row counts.

In [ ]:
rows = []
for p in sorted(DATA.glob('*.parquet')):
    pf = pq.ParquetFile(p)
    rows.append({
        'file': p.name,
        'rows': pf.metadata.num_rows,
        'cols': pf.metadata.num_columns,
        'row_groups': pf.metadata.num_row_groups,
        'size_MB': round(p.stat().st_size / 1e6, 2),
    })
pd.DataFrame(rows)

In [ ]:
def inspect(name, n_head=5, n_sample=0, describe=True):
    """Load a parquet and print a structural summary."""
    path = DATA / (name if name.endswith('.parquet') else f'{name}.parquet')
    df = pd.read_parquet(path)
    print(f'=== {path.name} ===')
    print(f'shape : {df.shape}')
    print(f'memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')
    print()
    info = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'nulls': df.isna().sum(),
        'null_%': (df.isna().mean() * 100).round(2),
        'nunique': [df[c].nunique(dropna=True) if df[c].dtype != 'object' or df[c].map(type).eq(str).all() else np.nan for c in df.columns],
    })
    display(info)
    print('\nhead:')
    display(df.head(n_head))
    if n_sample:
        print(f'\nsample({n_sample}):')
        display(df.sample(min(n_sample, len(df)), random_state=0))
    if describe:
        num = df.select_dtypes(include='number')
        if num.shape[1]:
            print('\nnumeric describe:')
            display(num.describe().T)
    return df

## Per-file inspection

Each call returns the DataFrame so you can keep poking. Comment out anything large you don't want in memory.

In [ ]:
address_codes = inspect('address_codes')

In [ ]:
blocks = inspect('blocks')

In [ ]:
clusters = inspect('clusters')

In [ ]:
receipts = inspect('receipts')

In [ ]:
token_transfers = inspect('token_transfers')

In [ ]:
transactions = inspect('transactions')

In [ ]:
wallet_features = inspect('wallet_features')

In [ ]:
wallet_features_scaled = inspect('wallet_features_scaled')

## Non-parquet artifacts

In [ ]:
with open(DATA / '_watermark.json') as f:
    watermark = json.load(f)
watermark

In [ ]:
with open(DATA / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
print(type(scaler).__module__ + '.' + type(scaler).__name__)
for attr in ('mean_', 'scale_', 'var_', 'n_features_in_', 'feature_names_in_', 'center_', 'data_min_', 'data_max_'):
    if hasattr(scaler, attr):
        v = getattr(scaler, attr)
        if isinstance(v, np.ndarray):
            print(f'{attr}: shape={v.shape}, dtype={v.dtype}')
        else:
            print(f'{attr}: {v}')

In [ ]:
umap = np.load(DATA / 'umap_emb.npy')
print('shape:', umap.shape, 'dtype:', umap.dtype)
print('range :', umap.min(axis=0), '→', umap.max(axis=0))
pd.DataFrame(umap[:5], columns=[f'd{i}' for i in range(umap.shape[1])])

## Cross-file sanity checks

Useful if you want to verify foreign-key-ish relationships. Edit column names to match your actual schema.

In [ ]:
# Example checks — tweak after you see the schemas above
checks = []

# wallet_features rows match clusters rows?
if len(wallet_features) == len(clusters):
    checks.append(('wallet_features rows == clusters rows', True, f'{len(wallet_features)}'))
else:
    checks.append(('wallet_features rows == clusters rows', False, f'{len(wallet_features)} vs {len(clusters)}'))

# umap_emb rows match wallet_features rows?
checks.append(('umap_emb rows == wallet_features rows', umap.shape[0] == len(wallet_features), f'{umap.shape[0]} vs {len(wallet_features)}'))

# scaled vs unscaled wallet_features same shape?
checks.append(('wallet_features shape == scaled shape', wallet_features.shape == wallet_features_scaled.shape, f'{wallet_features.shape} vs {wallet_features_scaled.shape}'))

pd.DataFrame(checks, columns=['check', 'ok', 'detail'])